In [1]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import spacy
import re
import contractions
from textblob import TextBlob

c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Load the document

In [2]:
data=open('data.txt',encoding='utf-8').read()

# 2. Text Normalization

### Converting all characters into lower

In [3]:
data=data.lower()

### Removing extra space

In [4]:
data=re.sub(r'\s{2,}','',data)

### Removing numbers like 1,2 3,4.....

In [5]:
# data=re.sub(r'\d+\.','',data)
# data

### Contractions

In [6]:
data=contractions.fix(data)

### Removing Special characters and punctuations

In [7]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)
data

'artificial intelligence ai is one of the fastestgrowing fields in technology it enables machines to simulate human intelligence and solve complex problems ai includes several domains such as machine learning deep learning natural language processing and computer visionmachine learning allows computers to learn from historical data and make predictions it can be divided into supervised learning unsupervised learning and reinforcement learning supervised learning uses labeled data while unsupervised learning finds hidden patterns in unlabeled datadeep learning is based on artificial neural networks it has revolutionized image recognition speech processing and language translation popular frameworks for deep learning include tensorflow and pytorchnatural language processing nlp enables computers to understand and generate human language nlp applications include chatbots sentiment analysis machine translation and text summarization tokenization stemming lemmatization and chunking are comm

### Textblob

In [8]:
# values=TextBlob(data).correct()
# values

### Spacy and Lemmatization

In [9]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)

updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(updated_tokens).strip()
data

'artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing computer visionmachine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning supervise learning use label datum unsupervised learning find hidden pattern unlabeled datadeep learning base artificial neural network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorchnatural language processing nlp enable computer understand generate human language nlp application include chatbot sentiment analysis machine translation text summarization tokenization stem lemmatization chunk common nlp preprocesse techniquescomputer vision focus enable machine interpret visual information facial recognition autonomous vehicle medical imaging surveillance system image

In [10]:
# tokens.ents

### Chunking(converted doc -> chunks)

In [11]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
chunks = splitter.create_documents([data])
chunks
print(chunks[0].page_content)

artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing


In [12]:
print(len(chunks))

214


In [13]:
chunks[0].metadata={'file_name':{'data.txt'}}
chunks

[Document(metadata={'file_name': {'data.txt'}}, page_content='artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing'),
 Document(metadata={}, page_content='deep learn natural language processing computer visionmachine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning'),
 Document(metadata={}, page_content='learning reinforcement learning supervise learning use label datum unsupervised learning find hidden pattern unlabeled datadeep learning base artificial neural network revolutionize image recognition'),
 Document(metadata={}, page_content='network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorchnatural language processing nlp enable computer understand'),
 Document(metadata={}, page_content

In [14]:
print(chunks)

[Document(metadata={'file_name': {'data.txt'}}, page_content='artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing'), Document(metadata={}, page_content='deep learn natural language processing computer visionmachine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning'), Document(metadata={}, page_content='learning reinforcement learning supervise learning use label datum unsupervised learning find hidden pattern unlabeled datadeep learning base artificial neural network revolutionize image recognition'), Document(metadata={}, page_content='network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorchnatural language processing nlp enable computer understand'), Document(metadata={}, page_content='nl

### Chunk Embeddings(converting chunks to vectors)

In [15]:
embeddings_model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2825.35it/s]


In [16]:
vectordb=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
)
vectordb

In [17]:
user_query='what is machine learning ?'
r_chunks=vectordb.similarity_search(user_query)

In [18]:
updated_r_chunks=set()
for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
updated_r_chunks
R_text='\n'.join(updated_r_chunks)
R_text

'network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorchnatural language processing nlp enable computer understand\ndeep learn natural language processing computer visionmachine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning\nlearning reinforcement learning supervise learning use label datum unsupervised learning find hidden pattern unlabeled datadeep learning base artificial neural network revolutionize image recognition\nartificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing'

In [19]:
def r_search(query,k=2):
        R_chunks=vectordb.similarity_search(query,k=k)
        R_chunks={doc.page_content for doc in R_chunks}
        R_text='\n'.join(R_chunks)
        return R_chunks
    
def g_text(r_search,query):
        import os
        prompt =f'''
              you are an helpfull assistant
              Assigned task for you: Structure my output => {r_search}
              for this input => {query}
            note:
            1)don't add extra contents just structure mentioned output.
            2)if there is any mistakes in output correct or else keep the original output with structured result
            with structured result.
            Output structure:
            Input:{query}
            Output:structure output'''
        
        llm_model=ChatGoogleGenerativeAI(
               model="gemini-3.5-flash",   #gemini-3.5-flash
               api_key=os.environ['gemini key']
          )
        response=llm_model.invoke(prompt).content 
        return response
    

user_prompt='Explain Machine Learning'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response =r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
g_response

[{'type': 'text',
  'text': 'Input:Explain Machine Learning\nOutput:\n* **Machine Learning:** Allows computers to learn from historical data to make predictions. It is divided into:\n  * **Supervised Learning:** Uses labeled data.\n  * **Unsupervised Learning:** Finds hidden patterns in unlabeled data.\n  * **Reinforcement Learning**\n* **Deep Learning:** Based on artificial neural networks, it has revolutionized fields like Natural Language Processing (NLP) and Computer Vision (including image recognition).',
  'extras': {'signature': 'EtwpCtkpARFNMg/l/BdgwiDXlcnlt8ABgkQdUObQdXhxuvmRbhImkGcrqtef/AfFXWIwFw/twnZcUbcu2cvhBQ74E9arrX0aORgjGi9NDaG+NctQebnDRfw2NliVUNA/BBBVFKzNkpBKy9a47NoDAKdveRzm2AKV3jsIY19QYNBNhZ3I+FTTCK7dzwff4j41pCxP5eE9fKfVpF/J7yIRK5uT6fYZv+QXDMYkxa9kt6OhCT2EaahYYnERQ3BZRmgfm3TtA55MtLesdkPKXcH73d+YEwE+P96+NK5HesSZPZ1Hjpz8J17HFJIED04qVT2jTt8znt9Fltprk8HIHhuoGmqpwOMGbojfHlx2PL41y7jELVjFEFvKykZEWzbpLtAzKgnIzUYOLkm1QUlRpo7x5Fedso1rEe9uYarTPrSGRtRSPTrCqNzgGgiSpjXkS05JcK1IHQ/mb

In [20]:
def r_search(query, k=2):
    docs = vectordb.similarity_search(query, k=k)

    retrieved_text = "\n".join([doc.page_content for doc in docs])

    return retrieved_text

def g_text(retrieved_text, query):
    prompt = f"""
You are a helpful assistant.

Assigned task:
Structure the retrieved output.

Retrieved Output:
{retrieved_text}

User Query:
{query}

Instructions:
1. Do not add extra information.
2. If there are grammatical or spelling mistakes, correct them.
3. Otherwise, keep the original content.
4. Present the result in the following format.

Output Format:

Input:
{query}

Output:
<Structured Output>
"""
    import os
    llm_model = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        api_key=os.environ['gemini key']
    )

    response = llm_model.invoke(prompt)

    return response.content


user_prompt = "Explain Machine Learning"

user_prompt = re.sub(r"[^0-9a-zA-Z\s]", "", user_prompt)

retrieved_text = r_search(user_prompt)

g_response = g_text(retrieved_text, user_prompt)

print(g_response)

[{'type': 'text', 'text': 'Input:\nExplain Machine Learning\n\nOutput:\n* **Machine Learning:** Allows computers to learn from historical data to make predictions. It is divided into:\n    * **Supervised Learning:** Uses labeled data.\n    * **Unsupervised Learning:** Finds hidden patterns in unlabeled data.\n    * **Reinforcement Learning**\n* **Associated Technologies (Deep Learning, Natural Language Processing, and Computer Vision):** \n    * Deep learning is based on artificial neural networks and has revolutionized image recognition.', 'extras': {'signature': 'EtJQCs9QARFNMg+GQh5sc6iI8bdDBmbBML8WEwQqr+do1Z1lhwSyni2+vsR5RRmAI90FmypvA8Z107aoxmD5KnypLmTXDYN1Z/mcYlpyMNehV1diF/ybb8uSEUSMZq0IdoKcsI2/ivJQMpMerN2VNIBepuySbNONtGTvRFlXWymfieyjWAKhbjWpZCLxPA76+4Z49K7P/UOW4Oifbn/5JMsEr9EVuHdAMKLGux2QF9UzrjTVoCmuPSnfG2bHYGH3n4OTt3Jw9Jo+PFhOGDp85aZy4yW+L3//mh20M00sUsIz2nqJaALhCEUZVdoxlyNZYzXc4W8p0e78bErrk4t1bCfFkk2fNIezpgl4byYpGie7/FxFP6F/L3JQ6EHai7nERGKlBRmSFIct3PgT6Td1TkQPYk9kqsPIBFfCJ9Wl9AKg